# 04 — Technical Specifications & Insurance (Team 4)

## Responsibility
This notebook handles all technical specification columns (engine, fuel consumption,
physical dimensions) and insurance cost columns.

## Columns Assigned
`Motor Hacmi`, `Motor Gücü`, `Ort. Yakıt Tüketimi`, `Yakıt Deposu`,
`Üretim Yılı (İlk/Son)`, `Ortalama Kasko`, `Ortalama Trafik Sigortası`,
`Silindir Sayısı`, `Tork`, `Maksimum Güç`, `Minimum Güç`,
`Hızlanma (0-100)`, `Maksimum Hız`,
`Ortalama Yakıt Tüketimi`, `Şehir İçi Yakıt Tüketimi`, `Şehir Dışı Yakıt Tüketimi`,
`Uzunluk`, `Genişlik`, `Yükseklik`, `Ağırlık`, `Boş Ağırlığı`,
`Koltuk Sayısı`, `Bagaj Hacmi`, `Ön Lastik`, `Aks Aralığı`

## Known Data Issues to Handle
- `Motor Hacmi`: some rows contain a range like "1401 - 1600 cm3" instead of exact value → take the midpoint; others contain "1461 cc" → strip "cc"
- `Motor Gücü`: same range issue e.g. "101 - 125 HP" → midpoint; strip "hp"
- `Ort. Yakıt Tüketimi`, `Ortalama Yakıt Tüketimi`, etc.: strip " lt", replace comma decimal separator with dot
- `Tork`: strip " nm"
- `Maksimum Güç`, `Minimum Güç`: strip " rpm"
- `Hızlanma (0-100)`: strip " sn", replace comma with dot
- `Maksimum Hız`: strip " km/s"
- `Uzunluk`, `Genişlik`, `Yükseklik`, `Aks Aralığı`: strip " mm"
- `Ağırlık`, `Boş Ağırlığı`: strip " kg"
- `Bagaj Hacmi`: strip " lt"
- `Yakıt Deposu`: strip " lt"
- `Ortalama Kasko`, `Ortalama Trafik Sigortası`: strip "TL" and dots → integer
- `Üretim Yılı (İlk/Son)`: contains ranges like "2014 - 2017" → extract start year as integer
- `Ön Lastik`: contains tire size strings like "225/45 R19" → extract the rim diameter number (19) as an integer feature named `Jant Boyutu`; drop original column
- All nulls filled with column median after unit stripping

## What is Expected as Output
All columns fully numeric (float or int), no nulls, no unit strings remaining.

## Input
raw_dataset.csv from GitHub raw URL

## Output
df_team4: a DataFrame slice containing all processed columns above,
with the same index as the original dataset (3424 rows)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RAW_URL = "https://raw.githubusercontent.com/MamoMGD1/ISE302-DataMining-GroupProject/main/data/raw_dataset.csv"
df_full = pd.read_csv(RAW_URL)
print(f"Loaded dataset: {df_full.shape}")
df_full.head(3)

In [ ]:
MY_COLUMNS = ['Motor Hacmi', 'Motor Gücü', 'Ort. Yakıt Tüketimi', 'Yakıt Deposu',
              'Üretim Yılı (İlk/Son)', 'Ortalama Kasko', 'Ortalama Trafik Sigortası',
              'Silindir Sayısı', 'Tork', 'Maksimum Güç', 'Minimum Güç',
              'Hızlanma (0-100)', 'Maksimum Hız', 'Ortalama Yakıt Tüketimi',
              'Şehir İçi Yakıt Tüketimi', 'Şehir Dışı Yakıt Tüketimi',
              'Uzunluk', 'Genişlik', 'Yükseklik', 'Ağırlık', 'Boş Ağırlığı',
              'Koltuk Sayısı', 'Bagaj Hacmi', 'Ön Lastik', 'Aks Aralığı']
df = df_full[MY_COLUMNS].copy()

## Step 1 — Null Analysis

In [ ]:
# Print null counts and percentage for each column
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)
print(pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct}))

In [ ]:
# Helper: parse a range string 'a - b <unit>' → midpoint float, or a single value → float
def parse_range_midpoint(series, strip_patterns):
    s = series.astype(str).str.strip()
    for pat in strip_patterns:
        s = s.str.replace(pat, '', regex=False, case=False).str.strip()
    # Replace comma decimal separators with dots
    s = s.str.replace(',', '.', regex=False)
    # Detect range pattern 'low - high'
    is_range = s.str.contains(r'^\d[\.\d]*\s*-\s*\d', regex=True, na=False)
    result = pd.to_numeric(s.where(~is_range), errors='coerce')
    # For range rows, extract low and high and take midpoint
    parts = s[is_range].str.extract(r'([\d\.]+)\s*-\s*([\d\.]+)')
    if not parts.empty:
        midpoints = (pd.to_numeric(parts[0], errors='coerce') +
                     pd.to_numeric(parts[1], errors='coerce')) / 2
        result[is_range] = midpoints
    return result

In [ ]:
# Clean Motor Hacmi: strip 'cm3' and 'cc', handle ranges → midpoint
df['Motor Hacmi'] = parse_range_midpoint(df['Motor Hacmi'], [' cm3', 'cm3', ' cc', 'cc'])
df['Motor Hacmi'] = df['Motor Hacmi'].fillna(df['Motor Hacmi'].median())
print(f"Motor Hacmi nulls: {df['Motor Hacmi'].isnull().sum()}")

In [ ]:
# Clean Motor Gücü: strip 'HP' and 'hp', handle ranges → midpoint
df['Motor Gücü'] = parse_range_midpoint(df['Motor Gücü'], [' HP', 'HP', ' hp', 'hp'])
df['Motor Gücü'] = df['Motor Gücü'].fillna(df['Motor Gücü'].median())
print(f"Motor Gücü nulls: {df['Motor Gücü'].isnull().sum()}")

In [ ]:
# Clean fuel consumption columns: strip ' lt', replace comma decimal separator
for col in ['Ort. Yakıt Tüketimi', 'Ortalama Yakıt Tüketimi',
            'Şehir İçi Yakıt Tüketimi', 'Şehir Dışı Yakıt Tüketimi']:
    df[col] = (
        df[col].astype(str)
        .str.replace(' lt', '', regex=False)
        .str.replace(',', '.', regex=False)
        .str.strip()
        .replace('nan', np.nan)
    )
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())
    print(f"{col} nulls: {df[col].isnull().sum()}")

In [ ]:
# Clean Yakıt Deposu: strip ' lt'
df['Yakıt Deposu'] = (
    df['Yakıt Deposu'].astype(str)
    .str.replace(' lt', '', regex=False)
    .str.strip()
    .replace('nan', np.nan)
)
df['Yakıt Deposu'] = pd.to_numeric(df['Yakıt Deposu'], errors='coerce')
df['Yakıt Deposu'] = df['Yakıt Deposu'].fillna(df['Yakıt Deposu'].median())
print(f"Yakıt Deposu nulls: {df['Yakıt Deposu'].isnull().sum()}")

In [ ]:
# Clean Tork: strip ' nm'
df['Tork'] = (
    df['Tork'].astype(str)
    .str.replace(' nm', '', regex=False, case=False)
    .str.strip()
    .replace('nan', np.nan)
)
df['Tork'] = pd.to_numeric(df['Tork'], errors='coerce')
df['Tork'] = df['Tork'].fillna(df['Tork'].median())
print(f"Tork nulls: {df['Tork'].isnull().sum()}")

In [ ]:
# Clean Maksimum Güç and Minimum Güç: strip ' rpm'
for col in ['Maksimum Güç', 'Minimum Güç']:
    df[col] = (
        df[col].astype(str)
        .str.replace(' rpm', '', regex=False, case=False)
        .str.strip()
        .replace('nan', np.nan)
    )
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())
    print(f"{col} nulls: {df[col].isnull().sum()}")

In [ ]:
# Clean Hızlanma (0-100): strip ' sn', replace comma with dot
df['Hızlanma (0-100)'] = (
    df['Hızlanma (0-100)'].astype(str)
    .str.replace(' sn', '', regex=False)
    .str.replace(',', '.', regex=False)
    .str.strip()
    .replace('nan', np.nan)
)
df['Hızlanma (0-100)'] = pd.to_numeric(df['Hızlanma (0-100)'], errors='coerce')
df['Hızlanma (0-100)'] = df['Hızlanma (0-100)'].fillna(df['Hızlanma (0-100)'].median())
print(f"Hızlanma (0-100) nulls: {df['Hızlanma (0-100)'].isnull().sum()}")

In [ ]:
# Clean Maksimum Hız: strip ' km/s'
df['Maksimum Hız'] = (
    df['Maksimum Hız'].astype(str)
    .str.replace(' km/s', '', regex=False)
    .str.strip()
    .replace('nan', np.nan)
)
df['Maksimum Hız'] = pd.to_numeric(df['Maksimum Hız'], errors='coerce')
df['Maksimum Hız'] = df['Maksimum Hız'].fillna(df['Maksimum Hız'].median())
print(f"Maksimum Hız nulls: {df['Maksimum Hız'].isnull().sum()}")

In [ ]:
# Clean dimension columns: strip ' mm'
for col in ['Uzunluk', 'Genişlik', 'Yükseklik', 'Aks Aralığı']:
    df[col] = (
        df[col].astype(str)
        .str.replace(' mm', '', regex=False)
        .str.strip()
        .replace('nan', np.nan)
    )
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())
    print(f"{col} nulls: {df[col].isnull().sum()}")

In [ ]:
# Clean weight columns: strip ' kg'
for col in ['Ağırlık', 'Boş Ağırlığı']:
    df[col] = (
        df[col].astype(str)
        .str.replace(' kg', '', regex=False)
        .str.strip()
        .replace('nan', np.nan)
    )
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())
    print(f"{col} nulls: {df[col].isnull().sum()}")

In [ ]:
# Clean Bagaj Hacmi: strip ' lt'
df['Bagaj Hacmi'] = (
    df['Bagaj Hacmi'].astype(str)
    .str.replace(' lt', '', regex=False)
    .str.strip()
    .replace('nan', np.nan)
)
df['Bagaj Hacmi'] = pd.to_numeric(df['Bagaj Hacmi'], errors='coerce')
df['Bagaj Hacmi'] = df['Bagaj Hacmi'].fillna(df['Bagaj Hacmi'].median())
print(f"Bagaj Hacmi nulls: {df['Bagaj Hacmi'].isnull().sum()}")

In [ ]:
# Clean Koltuk Sayısı: cast to numeric, fill nulls with median
df['Koltuk Sayısı'] = pd.to_numeric(df['Koltuk Sayısı'], errors='coerce')
df['Koltuk Sayısı'] = df['Koltuk Sayısı'].fillna(df['Koltuk Sayısı'].median())
print(f"Koltuk Sayısı nulls: {df['Koltuk Sayısı'].isnull().sum()}")

In [ ]:
# Clean Silindir Sayısı: cast to numeric, fill nulls with median
df['Silindir Sayısı'] = pd.to_numeric(df['Silindir Sayısı'], errors='coerce')
df['Silindir Sayısı'] = df['Silindir Sayısı'].fillna(df['Silindir Sayısı'].median())
print(f"Silindir Sayısı nulls: {df['Silindir Sayısı'].isnull().sum()}")

In [ ]:
# Clean insurance columns: strip 'TL' and dot thousands separators → integer
for col in ['Ortalama Kasko', 'Ortalama Trafik Sigortası']:
    df[col] = (
        df[col].astype(str)
        .str.replace('TL', '', regex=False)
        .str.replace('.', '', regex=False)
        .str.strip()
        .replace('nan', np.nan)
    )
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())
    print(f"{col} nulls: {df[col].isnull().sum()}")

In [ ]:
# Clean Üretim Yılı (İlk/Son): extract start year from ranges like '2014 - 2017'
df['Üretim Yılı (İlk/Son)'] = (
    df['Üretim Yılı (İlk/Son)']
    .astype(str)
    .str.extract(r'(\d{4})')[0]
)
df['Üretim Yılı (İlk/Son)'] = pd.to_numeric(df['Üretim Yılı (İlk/Son)'], errors='coerce')
df['Üretim Yılı (İlk/Son)'] = df['Üretim Yılı (İlk/Son)'].fillna(df['Üretim Yılı (İlk/Son)'].median()).astype(int)
print(f"Üretim Yılı (İlk/Son) nulls: {df['Üretim Yılı (İlk/Son)'].isnull().sum()}")

In [ ]:
# Clean Ön Lastik: extract rim diameter from tire size strings like '225/45 R19' → Jant Boyutu
df['Jant Boyutu'] = (
    df['Ön Lastik']
    .astype(str)
    .str.extract(r'R(\d+)')[0]
)
df['Jant Boyutu'] = pd.to_numeric(df['Jant Boyutu'], errors='coerce')
df['Jant Boyutu'] = df['Jant Boyutu'].fillna(df['Jant Boyutu'].median()).astype(int)
# Drop original Ön Lastik column
df.drop(columns=['Ön Lastik'], inplace=True)
print(f"Jant Boyutu nulls: {df['Jant Boyutu'].isnull().sum()}")

In [ ]:
df_team4 = df.copy()
assert df_team4.isnull().sum().sum() == 0, "Nulls remain!"
assert df_team4.select_dtypes(exclude='number').shape[1] == 0, "Non-numeric columns remain!"
print(f"✅ Team 4 output ready. Shape: {df_team4.shape}")
print(df_team4.dtypes)